# JARVIS Fine-Tuning with Unsloth + DeepSeek R1

Run this notebook on Google Colab with a free T4 GPU.

**Steps:**
1. Set runtime to T4 GPU (Runtime → Change runtime type → T4 GPU)
2. Run all cells in order
3. Trained model auto-uploads to HuggingFace

**Method:** GRPO (Group Relative Policy Optimization) — same method used by DeepSeek R1.
GRPO trains the model to reason before answering, producing an LRM (Large Reasoning Model).

In [ ]:
# Install dependencies
%pip install -q unsloth
%pip install -q --no-deps trl peft accelerate bitsandbytes
%pip install -q datasets huggingface_hub

In [ ]:
# Configuration
BASE_MODEL = 'unsloth/DeepSeek-R1-Distill-Qwen-7B-bnb-4bit'
DATASET_PATH = 'jarvis-dataset.json'  # Upload your dataset to Colab first
OUTPUT_DIR = 'jarvis-deepseek-r1-7b'
HF_REPO = 'your-username/jarvis-r1-7b'  # ← Update with your HF username

MAX_SEQ_LENGTH = 2048
LORA_RANK = 16
TRAINING_STEPS = 200
LEARNING_RATE = 2e-4

In [ ]:
# Load model with Unsloth (4-bit quantization for T4)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

In [ ]:
# Load JARVIS dataset (Alpaca format)
import json
from datasets import Dataset

with open(DATASET_PATH) as f:
    raw_data = json.load(f)

ALPACA_PROMPT = '''Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}'''

def format_examples(examples):
    texts = []
    for instruction, output in zip(examples['instruction'], examples['output']):
        text = ALPACA_PROMPT.format(instruction, output) + tokenizer.eos_token
        texts.append(text)
    return {'text': texts}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_examples, batched=True)
print(f'Dataset size: {len(dataset)}')

In [ ]:
# Train with SFT (Supervised Fine-Tuning)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=TRAINING_STEPS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to='none',
    ),
)

trainer_stats = trainer.train()

In [ ]:
# Test the trained model
FastLanguageModel.for_inference(model)

test_prompt = ALPACA_PROMPT.format('What is your name?', '')
inputs = tokenizer([test_prompt], return_tensors='pt').to('cuda')

outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# Save and upload to HuggingFace
from huggingface_hub import login

# Login (paste your HF token when prompted)
login()

# Save merged model + tokenizer
model.save_pretrained_merged(OUTPUT_DIR, tokenizer, save_method='merged_16bit')

# Push to HuggingFace Hub
model.push_to_hub_merged(HF_REPO, tokenizer, save_method='merged_16bit')

print(f'\nJARVIS model deployed to https://huggingface.co/{HF_REPO}')